# Fineturner ResNet pour une LiveNess Detection

Ce notebook est fait pour détailler le funeturning de __CelebA-Spoof__


In [1]:
print("Hello")

Hello


## Import des bibliothèques

In [6]:
import os
import io 
import zipfile
from pathlib import Path
from typing import Dict, List, Tuple, cast
from PIL import Image

import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset , Subset
from torchvision import datasets, models, transforms

## Liste des dossiers disponible

In [3]:
for dirname, _, filenames in os.walk('/kaggle/input'):
    print(dirname)
    if len(filenames) > 0:
        print(f"  -> Contient {len(filenames)} fichiers/images")

/kaggle/input
/kaggle/input/datasets
/kaggle/input/datasets/immada
/kaggle/input/datasets/immada/celeba-spoof-crop
/kaggle/input/datasets/immada/celeba-spoof-crop/CelebA_Spoof
/kaggle/input/datasets/immada/celeba-spoof-crop/CelebA_Spoof/test
/kaggle/input/datasets/immada/celeba-spoof-crop/CelebA_Spoof/test/live
  -> Contient 19923 fichiers/images


KeyboardInterrupt: 

# Chargement des poids de ResNet
Ici je personnalise la tête de ResNet et l'achitecture finale devient
2048 --> 256 --> 2 

In [4]:
def create_model(num_classes: int = 2, pretrained: bool = True) -> nn.Module:
    weights = models.ResNet18_Weights.IMAGENET1K_V1 if pretrained else None
    model = models.resnet18(weights=weights)
    
    # Geler tout le réseau de base si nécéssaire
    if pretrained :
        for param in model.parameters():
            param.requires_grad = False
    
    # Tête multi-couches avec Dropout
    model.fc = nn.Sequential(
        nn.Linear(model.fc.in_features, 256),
        nn.ReLU(),
        nn.Dropout(0.3),
        nn.Linear(256, num_classes)
    )
    
    return model

## Chargement du DataSet depuis des fichier zip

In [ ]:
class ZipDataset(Dataset):

  def __init__(self, zip_path: str, split: str = 'train', transform=None):
    """Dataset PyTorch lisant directement depuis un fichier ZIP.

    Args:
        zip_path (str): Chemin vers le fichier zip.
        split (str): 'train' ou 'test'.
        transform (callable, optional): Transformations PyTorch à appliquer.
    """
    self.zip_path = zip_path
    self.split = split
    self.transform = transform

  
    # 'live' -> 0, 'spoof' -> 1 
    self.class_to_idx = {'live': 0, 'spoof': 1}

    # Pré-requis d'extensions valides
    valid_extensions = (
        '.jpg',
        '.jpeg',
        '.png',
        '.bmp',
        '.JPG',
        '.JPEG',
        '.PNG',
    )

    # Construction du préfixe exact attendu dans le zip
    # Ex: "Spoof_Live/CelebA_Spoof/train/"
    self.prefix = f'CelebA_Spoof/{split}/'

    self.image_paths = []
    self.labels = []
    
    self.zip_file = zipfile.ZipFile(self.zip_path, 'r')
    
    for file_path in self.zip_file.namelist():
      if file_path.startswith('__MACOSX') or not file_path.endswith(valid_extensions):
        continue

      parts = file_path.split('/')

      if self.split in parts:
        split_idx = parts.index(self.split)
        if split_idx + 1 < len(parts):
          class_name = parts[split_idx + 1]
          if class_name in self.class_to_idx:
            self.image_paths.append(file_path)
            self.labels.append(self.class_to_idx[class_name])
      
    

  def __len__(self):
    return len(self.image_paths)

  def __getitem__(self, idx):
    file_path = self.image_paths[idx]
    label = self.labels[idx]

    # Lecture de l'image binaire depuis l'archive sans extraction sur disque
    with zipfile.ZipFile(self.zip_path, 'r') as z:
      img_bytes = z.read(file_path)
      image = Image.open(io.BytesIO(img_bytes)).convert('RGB')

    # Application du prétraitement
    if self.transform is not None:
      image = self.transform(image)

    return image, label

## Transformations à appliquer au Images

In [3]:
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]

def build_transforms(train: bool = False, image_size: int = 224) -> transforms.Compose:
    if train:
        return transforms.Compose(
            [
                transforms.RandomResizedCrop(image_size),
                transforms.RandomHorizontalFlip(),
                # Variations de couleur
                transforms.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.3, hue=0.1),
                transforms.ToTensor(),
                transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
                # Effacer aléatoirement une petite zone 
                # et moins dépendant des artefacts locaux.
                transforms.RandomErasing(p=0.3, scale=(0.02, 0.1)),
            ]
        )
    return transforms.Compose(
        [
            transforms.Resize(int(image_size * 1.14)),
            transforms.CenterCrop(image_size),
            transforms.ToTensor(),
            transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
        ]
    )

## DataLoader

In [7]:
DATASET_ROOT = "/kaggle/input/datasets/immada/celeba-spoof-crop/CelebA_Spoof"
TRAIN_DIR = os.path.join(DATASET_ROOT, 'train')
TEST_DIR = os.path.join(DATASET_ROOT, 'test')

subset_size = 20000

print("Image folder for Train")
train_dataset = datasets.ImageFolder(root=TRAIN_DIR, transform=build_transforms(train=True))
indices = np.random.choice(len(train_dataset), subset_size, replace=False)
train_dataset = Subset(train_dataset, indices)


print("Image Folder for Test")
test_dataset = datasets.ImageFolder(root=TEST_DIR, transform=build_transforms(train = False))
indices = np.random.choice(len(test_dataset), subset_size, replace=False)
test_dataset = Subset(test_dataset, indices)

print("Data Loader for train")
train_loader = DataLoader(train_dataset, batch_size=64,num_workers=4, pin_memory=True, shuffle=True)
print("DataLoaader for Test")
test_loader   = DataLoader(test_dataset, batch_size=64,num_workers=4,pin_memory=True, shuffle=False)

print("dataloaders ready")

Image folder for Train
Image Folder for Test
Data Loader for train
DataLoaader for Test
dataloaders ready


In [11]:
def evaluate(model, loader, criterion, device):
    model.eval()
    correct, total, running_loss = 0, 0, 0.0
    with torch.no_grad():
        for images, labels in loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            loss = criterion(outputs, labels)
            running_loss += loss.item()
            preds = outputs.argmax(dim=1)
            correct += (preds == labels).sum().item()
            total += labels.size(0)
    return running_loss / len(loader), correct / total

In [8]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device : {device}")
model = create_model()

model = model.to(device)
criterion = nn.CrossEntropyLoss()

Device : cuda
Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /root/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth


100%|██████████| 44.7M/44.7M [00:00<00:00, 230MB/s]


In [15]:
print(" Entraînement de la tête personnalisée ")

optimizer_head = torch.optim.Adam(model.fc.parameters(), lr=1e-3)

for epoch in range(4): 
    print(f"Epoch {epoch+1}")
    model.train()
    running_loss = 0.0
    
    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)
        
        optimizer_head.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer_head.step()
        
        running_loss += loss.item()
    print("Train")   
    print(f"Époque {epoch+1}/4 | Loss : {running_loss / len(train_loader):.4f}")

    # Evaluation
    print("Test")
    evaluation = evaluate(model , test_loader , criterion , device)
    print(f"Époque {epoch+1}/4 | Loss : {evaluation[0]}:.4f ")
    print(f"Accuracy : {evaluation[1]} ")


 Entraînement de la tête personnalisée 
Epoch 1
Train
Époque 1/4 | Loss : 0.4165
Test
Époque 1/4 | Loss : 0.5498211015337191:.4f 
Accuracy : 0.7158 
Epoch 2
Train
Époque 2/4 | Loss : 0.4046
Test
Époque 2/4 | Loss : 0.712835636668312:.4f 
Accuracy : 0.6296 
Epoch 3
Train
Époque 3/4 | Loss : 0.4063
Test
Époque 3/4 | Loss : 0.6837161782260139:.4f 
Accuracy : 0.6261 
Epoch 4
Train
Époque 4/4 | Loss : 0.4054
Test
Époque 4/4 | Loss : 0.7675453640591984:.4f 
Accuracy : 0.6243 


In [17]:
import torch

# Enregistrement
torch.save(model, "model_live_spoof.pth")


In [16]:

print("\n--- PHASE 2 : Dégal du bloc 'layer4' et Fine-tuning ---")

# Dégeler le dernier bloc de convolutions
for param in model.layer4.parameters():
    param.requires_grad = True

# Optimiseur avec un Learning Rate plus petit pour le corps, plus grand pour la tête
optimizer_fine = torch.optim.Adam([
    {'params': model.layer4.parameters(), 'lr': 1e-4},
    {'params': model.fc.parameters(),     'lr': 1e-3}
])

for epoch in range(5):  # Ré-entraînement global
    model.train()
    running_loss = 0.0
    
    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)
        
        optimizer_fine.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer_fine.step()
        
        running_loss += loss.item()
    print("Train") 
    print(f"Époque {epoch+1}/5 | Loss : {running_loss / len(train_loader):.4f}")
    print("Test")
    evaluation = evaluate(model , test_loader , criterion , device)
    print(f"Époque {epoch+1}/4 | Loss : {evaluation[0]:.4f} ")
    print(f"Accuracy : {evaluation[1]:.4f} ")


--- PHASE 2 : Dégal du bloc 'layer4' et Fine-tuning ---
Train
Époque 1/5 | Loss : 0.3437
Test
Époque 1/4 | Loss : 1.0186 
Accuracy : 0.6218 
Train
Époque 2/5 | Loss : 0.2874
Test
Époque 2/4 | Loss : 0.5922 
Accuracy : 0.7607 
Train
Époque 3/5 | Loss : 0.2609
Test
Époque 3/4 | Loss : 0.9173 
Accuracy : 0.6550 
Train
Époque 4/5 | Loss : 0.2544
Test
Époque 4/4 | Loss : 0.9320 
Accuracy : 0.6539 
Train
Époque 5/5 | Loss : 0.2366
Test
Époque 5/4 | Loss : 1.0314 
Accuracy : 0.7003 
